In [185]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import RidgeCV
from sklearn.linear_model import LassoCV
from sklearn.linear_model import ElasticNetCV
from sklearn.model_selection import GridSearchCV
from dataclasses import dataclass
from sklearn.base import BaseEstimator, TransformerMixin
from tqdm.notebook import tqdm
from itertools import product



In [186]:
@dataclass(frozen=True)
class Path:
    TRAIN_DIR: str = "kaggle/input/hull-tactical-market-prediction/train.csv"
    TEST_DIR: str = "/kaggle/input/hull-tactical-market-prediction/test.cv"
    MODELS_DIR: str = "/kaggle/working/models"
    OOF_DIR: str = "/kaggle/working/oof"
    METRICS_CSV: str = "/kaggle/working/metrics.csv"

In [187]:
df_raw = pd.read_csv(Path.TRAIN_DIR)
EXCLUDED_COLS = {
    'date_id',
    'forward_returns',
    'risk_free_rate',
    'market_forward_excess_returns',
}
DATE_COL = 'date_id'

TARGET_COL = 'market_forward_excess_returns'

PREFIX_BUCKETS = {
    "M":   "Market",
    "E":   "Macro",
    "I":   "Interest",
    "P":   "Price",
    "V":   "Volatility",
    "S":   "Sentiment",
    "D":   "Dummy",
}


In [188]:
class LaggedColumnSynthesizer(BaseEstimator, TransformerMixin):
    def __init__(self, date_col="date_id", sentinel=0):
        self.date_col = date_col
        self.sentinel = sentinel

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        df = X.copy()

        if self.date_col in df.columns:
            df = df.sort_values(self.date_col).reset_index(drop=True)

        if "lagged_forward_returns" not in df.columns and "forward_returns" in df.columns:
            df["lagged_forward_returns"] = df["forward_returns"].shift(1)

        if "lagged_risk_free_rate" not in df.columns and "risk_free_rate" in df.columns:
            df["lagged_risk_free_rate"] = df["risk_free_rate"].shift(1)

        if "lagged_market_forward_excess_returns" not in df.columns and "market_forward_excess_returns" in df.columns:
            df["lagged_market_forward_excess_returns"] = df["market_forward_excess_returns"].shift(1)

        for c in ["lagged_forward_returns", "lagged_risk_free_rate", "lagged_market_forward_excess_returns"]:
            if c in df.columns:
                df[c] = df[c].replace([np.inf, -np.inf], np.nan)

        return df

In [189]:
class LaggedTargetMomentum(BaseEstimator, TransformerMixin):
    """
    Build momentum features from a PRE-LAGGED target column.

    Assumptions:
    - `target_col` already contains past returns (e.g. market_forward_excess_returns.shift(1)).
    - This transformer never sees the true contemporaneous target.
    """

    def __init__(
        self,
        target_col,
        use_log=True,
        sentinel=0,
        windows=(5, 10, 21, 63, 126, 252),
        lags=(1, 2, 3, 5, 10, 21, 63, 126),
        drop_source=True,
    ):
        self.target_col = target_col
        self.use_log = use_log
        self.sentinel = sentinel
        self.windows = windows
        self.lags = lags
        self.drop_source = drop_source

    def fit(self, X, y=None):
        self._use_log = bool(self.use_log)
        self._sentinel = float(self.sentinel)
        self._windows = tuple(self.windows)
        self._lags = tuple(self.lags)
        return self

    def _base_series(self, df: pd.DataFrame) -> pd.Series:
        """Return cleaned, optionally log-transformed PRE-LAGGED series."""
        if self.target_col not in df.columns:
            raise KeyError(f"Target column '{self.target_col}' not found in DataFrame.")

        s = pd.to_numeric(df[self.target_col], errors="coerce")
        s = s.replace([np.inf, -np.inf], np.nan)
        s = s.mask(s == self._sentinel, np.nan)

        if not self._use_log:
            return s

        eps = 1e-12
        v = s.to_numpy(dtype=float)
        out = np.full_like(v, np.nan)
        finite = np.isfinite(v)

        vc = v.copy()
        vc[finite] = np.maximum(vc[finite], -1.0 + eps)

        safe = np.zeros_like(v, dtype=bool)
        safe[finite] = vc[finite] > -1.0

        out[safe] = np.log1p(vc[safe])
        return pd.Series(out, index=s.index)

    def transform(self, X):
        df = X.copy()
        y_lag = self._base_series(df)   # already past-only by construction

        feats = {}

        # rolling windows of PAST returns
        for window in self._windows:
            roll = y_lag.rolling(window, min_periods=max(5, window // 5))
            feats[f"MOM_y_mean_{window}"] = roll.mean()
            feats[f"MOM_y_std_{window}"] = roll.std()
            feats[f"MOM_y_roc_{window}"] = y_lag / y_lag.shift(window) - 1
            feats[f"MOM_y_cum_{window}"] = roll.sum()

        # EMAs
        ema_fast = y_lag.ewm(span=10, adjust=False).mean()
        ema_slow = y_lag.ewm(span=50, adjust=False).mean()
        feats["MOM_y_ema_fast"] = ema_fast
        feats["MOM_y_ema_slow"] = ema_slow
        feats["MOM_y_ema_diff"] = ema_fast - ema_slow
        with np.errstate(divide="ignore", invalid="ignore"):
            feats["MOM_y_ema_ratio"] = ema_fast / ema_slow - 1

        # vol-of-vol ratio if available
        if "MOM_y_std_21" in feats and "MOM_y_std_252" in feats:
            with np.errstate(divide="ignore", invalid="ignore"):
                feats["MOM_y_std_21_252_ratio"] = feats["MOM_y_std_21"] / feats["MOM_y_std_252"]

        # extra lags of the (already lagged) series
        for lag in self._lags:
            feats[f"MOM_y_lag_{lag}"] = y_lag.shift(lag - 1)

        feats_df = (
            pd.DataFrame(feats, index=df.index)
            .replace([np.inf, -np.inf], np.nan)
        )

        # optionally drop the source lagged series if you don't want it as a raw feature
        if self.drop_source and self.target_col in df.columns:
            df = df.drop(columns=[self.target_col])

        return pd.concat([df, feats_df], axis=1)

In [190]:
class FeatureBuilder(BaseEstimator, TransformerMixin):
    def __init__(self,
                 sentinel=0, 
                 windows=(5,10,21,63,126,252),
                 lags=(1, 2, 3, 5, 10, 21, 63, 126),
                 dummy_lags=(1, 2, 5, 10 , 21),
                 dummy_windows=(5, 21),
                 clip_roc_extremes=True,
                 roc_clip=10,
                 excluded_cols=None,
                 group_prefix_bucket=None,
                 verbose: bool = False):
        self.sentinel = sentinel
        self.windows = windows
        self.lags = lags
        self.dummy_lags = dummy_lags
        self.dummy_windows = dummy_windows
        self.clip_roc_extremes = clip_roc_extremes
        self.roc_clip = roc_clip
        self.excluded_cols = excluded_cols
        self.group_prefix_bucket = group_prefix_bucket
        self.verbose = verbose

    def fit(self, X, y=None):
        self._sentinel = float(self.sentinel)
        self._windows = tuple(self.windows)
        self._lags = tuple(self.lags)
        self._dummy_lags = tuple(self.dummy_lags)
        self._dummy_windows = tuple(self.dummy_windows)
        self._clip_roc_extremes = bool(self.clip_roc_extremes)
        self._roc_clip = float(self.roc_clip)
        self._excluded_cols = set(self.excluded_cols) if self.excluded_cols is not None else set()
        self._prefix_buckets = dict(self.group_prefix_bucket) if self.group_prefix_bucket is not None else dict(PREFIX_BUCKETS)

        self._groups = self._create_category_groups(X)
        if self.verbose:
            counts = {k: len(v) for k, v in self._groups.items()}
            print("[FeatureBuilder] groups:", counts)
        return self

    def transform(self, X):
        df = X.copy()
        df = self._sanitize_numeric(df)

        continuous_buckets = [
            b for b in self._groups.keys()
            if b not in ("Momentum", "Lagged", "Excluded", "Dummy", "Volatility Indicator")
        ]

        cont_cols = []
        for b in continuous_buckets:
            cont_cols.extend(self._groups.get(b, []))

        # do NOT touch MOM_ / VI_ / lagged_ or excluded
        cont_cols = [
            c for c in cont_cols
            if not c.startswith(("MOM", "VI", "lagged_")) and c not in self._excluded_cols
        ]

        if cont_cols:
            df, _ = self._extend_features(df, cols=cont_cols)

        d_cols = self._groups.get("Dummy", [])
        if d_cols:
            df = self._extend_binaries(df, cols=d_cols)

        if self._clip_roc_extremes:
            roc_cols = [c for c in df.columns if "_roc_" in c]
            if roc_cols:
                df.loc[:, roc_cols] = df.loc[:, roc_cols].clip(-self._roc_clip, self._roc_clip)

        return df

    def _sanitize_numeric(self, df):
        out = df.copy()
        feat_cols = [c for c in out.columns if c not in self._excluded_cols]
        out[feat_cols] = out[feat_cols].replace([np.inf, -np.inf], np.nan)
        return out

    def _create_category_groups(self, df: pd.DataFrame) -> dict:
        groups = {
            "Market": [], "Macro": [], "Interest": [], "Price": [],
            "Volatility": [], "Sentiment": [], "Dummy": [],
            "Momentum": [], "Lagged": [], "Volatility Indicator": [],
            "Excluded": [], "Other": []
        }
        for c in df.columns:
            if c in self._excluded_cols:
                groups["Excluded"].append(c)
                continue
            if c.startswith("lagged_"):
                groups["Lagged"].append(c)
                continue
            if c.startswith("MOM"):
                groups["Momentum"].append(c)
                continue
            if c.startswith("VI"):
                groups["Volatility Indicator"].append(c)
                continue

            p = c[:1]
            bucket = self._prefix_buckets.get(p)
            if bucket is not None:
                groups[bucket].append(c)
            else:
                groups["Other"].append(c)
        return groups

    def _impute_independent_variables(self, df: pd.DataFrame) -> pd.DataFrame:
        df_out = df.copy()

        feature_cols = [col for col in df_out.columns if col not in self.excluded_cols]
        flag_cols = []
    
        for col in feature_cols:
            if df_out[col].isnull().any():
                df_out[f"{col}_is_missing"] = df_out[col].isnull().astype("int8")

    
        df_out[feature_cols] = (df_out[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(sentinal))
        
        return df_out

    def _extend_all_features(self, df: pd.DataFrame, category_groups: dict) -> pd.DataFrame:
        df_out = df.copy()

        for category, cols in category_groups.items():
            print(category)
            
            if category in ("MOM", "VI") or category.startswith("lagged"):
                continue
            if group == "D":
                df_out, new_cols = self._extend_binaries(df_out, d_cols)
            else:
                df_out, new_cols = self._extend_features(df_out, sentinel=self.sentinel, cols=self.category_groups[category])

            print(f"Added {len(new_cols)} new {category} features")

        return df_out

    def _extend_features(self, df: pd.DataFrame,
                         sentinel=None,
                         cols=None,
                         windows=None, lags=None):
        windows = windows or self.windows
        lags = lags or self.lags
        sentinel = sentinel or self.sentinel
        cols = cols or self.category_groups[category]
        
        df = df.copy()
        feat_frames = []
        new_cols = []
    
        for col in [c for c in cols if not c.startswith(('MOM', 'VI'))]:
                col_raw = df[col].astype(float)
                col_shifted = col_raw.shift(1)
    
                category_extensions = {}
                
                for window in windows:
                    roll = col_shifted.rolling(window, min_periods=max(5, window // 5))
                    category_extensions[f'{col}_mean_{window}'] = roll.mean()
                    category_extensions[f'{col}_std_{window}'] = roll.std()
                    category_extensions[f'{col}_roc_{window}'] = col_shifted / col_shifted.shift(window) - 1
                    category_extensions[f'{col}_cum_{window}'] = roll.sum()
    
                for lag in lags:
                    category_extensions[f'{col}_lag_{lag}'] = col_shifted.shift(lag-1)
    
                ema_fast = col_shifted.ewm(span=10, adjust=False).mean()
                ema_slow = col_shifted.ewm(span=50, adjust=False).mean()
                category_extensions[f'{col}_ema_fast'] = ema_fast
                category_extensions[f'{col}_ema_slow'] = ema_slow
                category_extensions[f'{col}_ema_diff'] = ema_fast - ema_slow
                with np.errstate(divide="ignore", invalid="ignore"):
                    category_extensions[f'{col}_ema_ratio'] = ema_fast / ema_slow - 1
    
                feat_df = pd.DataFrame(category_extensions, index=df.index).replace([np.inf, -np.inf], np.nan).fillna(sentinel)
                feat_frames.append(feat_df)
                new_cols.extend(feat_df.columns.tolist())

        if feat_frames:
            feats = pd.concat(feat_frames, axis=1)
            df_out = pd.concat([df, feats], axis=1)
        else:
            df_out = df
            
        return df_out, new_cols

    def _extend_binaries(self, df: pd.DataFrame, cols):
        frames = []
        for col in cols:
            s = df[col].astype(float).shift(1)
            ext = {}
            # short lags
            for k in self.dummy_lags:
                ext[f"{col}_lag_{k}"] = s.shift(k)
            # short rolling mean as "recent on-rate"
            for w in self.dummy_windows:
                ext[f"{col}_mean_{w}"] = s.rolling(w, min_periods=1).mean()
            fr = (pd.DataFrame(ext, index=df.index)
                    .replace([np.inf, -np.inf], np.nan)
                    .fillna(self.sentinel))
            frames.append(fr)

        if frames:
            extra = pd.concat(frames, axis=1)
            return pd.concat([df, extra], axis=1)
        return df

In [191]:
class VolatilityIndicators(BaseEstimator, TransformerMixin):
    def __init__(self, 
                 lag_col="lagged_target",
                 use_log=True,
                 sentinel=0,
                 highvol_ratio=1.2,
                 shock_z=2.5,
                 bull_sma=200,
                 fallback_from_raw=True):
        self.lag_col = lag_col
        self.use_log = use_log
        self.sentinel = sentinel
        self.highvol_ratio = highvol_ratio
        self.shock_z = shock_z
        self.bull_sma = bull_sma
        self.fallback_from_raw = fallback_from_raw
        
    def fit(self, X, y=None):
        self._use_log = bool(self.use_log)
        self._sentinel = float(self.sentinel)
        self._fallback = bool(self.fallback_from_raw)
        self._lag_col = self.lag_col
        return self
            
    def _get_lag_series(self, df):
        # 1) choose source: lag col if present, else raw shifted
        if self._lag_col in df:
            s = pd.to_numeric(df[self._lag_col], errors="coerce")
        else:
            return None
    
        # 2) sanitize
        s = s.replace([np.inf, -np.inf], np.nan)
        if hasattr(self, "_sentinel"):
            s = s.mask(s == self._sentinel, np.nan)
    
        if not self._use_log:
            return s
    
        # 3) finite-only lower bound and finite-only comparison to avoid FloatingPointError
        eps = 1e-12
        v = s.to_numpy(dtype=float)
        out = np.full_like(v, np.nan)
        finite = np.isfinite(v)
    
        vc = v.copy()
        vc[finite] = np.maximum(vc[finite], -1.0 + eps)
    
        safe = np.zeros_like(v, dtype=bool)     # build mask without touching NaNs
        safe[finite] = vc[finite] > -1.0        # comparison only on finite entries
    
        out[safe] = np.log1p(vc[safe])          # raw np.log1p, zero drama
        return pd.Series(out, index=s.index)

    
    def transform(self, X):
        df = X.copy()
        y_lag = self._get_lag_series(df)
        
        feat = {}

        std21_252_ratio = df.get("MOM_y_std_21_252_ratio")
        if std21_252_ratio is not None:
            feat["VI_regime_highvol"] = (std21_252_ratio.fillna(-np.inf) > self.highvol_ratio).astype("int8")
        else:
            std21 = df.get("MOM_y_std_21")
            std252 = df.get("MOM_y_std_252")
            if std21 is not None and std252 is not None:
                with np.errstate(divide="ignore", invalid="ignore"):
                    ratio = std21 / std252
                feat["VI_regime_highvol"] = (ratio > self.highvol_ratio).astype("int8")
            else:
                feat["VI_regime_highvol"] = pd.Series(0, index=df.index, dtype="int8")

        if not y_lag.empty:
            den = y_lag.rolling(21, min_periods=5).std()
            with np.errstate(divide="ignore", invalid="ignore"):
                z = y_lag / den
            feat["VI_regime_shock"] = (z.abs().fillna(-np.inf) > 2.5).astype("int8")
        else:
            feat["VI_regime_shock"] = pd.Series(0, index=df.index, dtype="int8")

        if not y_lag.empty:
            r = np.expm1(y_lag.fillna(0)) if self.use_log else y_lag.fillna(0)
            mkt = (1 + r).cumprod()
            sma200 = mkt.rolling(self.bull_sma, min_periods=20).mean()
            feat["VI_regime_bull"] = (mkt.fillna(-np.inf) > sma200.fillna(-np.inf)).astype("int8")
        else:
            feat["VI_regime_bull"] = pd.Series(0, index=df.index, dtype="int8")

        std21 = df.get("MOM_y_std_21")
        if std21 is not None:
            feat["VI_volofvol_21"] = std21.pct_change().fillna(0).replace([np.inf, -np.inf], 0)
        else:
            feat["VI_volofvol_21"] = pd.Series(0.0, index=df.index)

        for col in ["MOM_y_ema_diff", "MOM_roc_21", "MOM_mean_21"]:
            if col in df:
                feat[f"VI_{col}_x_bull"] = df[col]*feat["VI_regime_bull"]
                feat[f"VI_{col}_x_highvol"] = df[col]*feat["VI_regime_highvol"]

        ema_diff = df.get("MOM_y_ema_diff")
        if ema_diff is not None:
            feat["VI_regime_highvol_x_ema"] = feat["VI_regime_highvol"] * ema_diff
        else:
            feat["VI_regime_highvol_x_ema"] = pd.Series(0.0, index=df.index)

        feat_df = pd.DataFrame(feat, index=df.index) \
                    .replace([np.inf, -np.inf], np.nan)
        df_out = pd.concat([df, feat_df], axis=1)

        return df_out

In [192]:
def make_base_target(df):
    df = df.sort_values(DATE_COL).reset_index(drop=True)
    df["lagged_target"] = df[TARGET_COL].shift(1)
    df = df.dropna(subset=["lagged_target"])
    y = df[TARGET_COL].astype(float)
    return df, y

def impute_simple(df):
    df = df.copy()
    return df.fillna(0)

def impute_medium(df, cutoff=1006):
    df = df.iloc[cutoff:].copy()
    df = df.fillna(0)
    return df

def impute_advanced(df, cutoff=1006):
    df = df.iloc[cutoff:].copy()
    df = df.copy()
    miss_flags = df.isna().astype("int8").add_suffix("_is_missing")
    df = df.fillna(0)
    return pd.concat([df, miss_flags], axis=1)


IMPUTERS = {
    "simple": impute_simple,
    "medium": impute_medium,
    "advanced": impute_advanced,
}

In [193]:
FEATURE_GROUPS = {
    1: ["E7", "M1", "M13", "M14", "M6", "S3", "V10", "V9"],
    2: ["M2", "M5", "S12", "S8"],
    3: ["E1", "E20", "M3", "P5", "P6", "P7", "S5", "V13", "V5", "V7"],
}

def drop_feature_groups(X_df: pd.DataFrame, level: int) -> pd.DataFrame:
    """
    Drop progressively more features based on 'level' in [1,4].
    If level = k, drop all columns from groups 1..k.
    Safely ignores columns that aren't present.
    """
    if level is None or level <= 0:
        return X_df

    cols_to_drop = []
    for g in range(1, level + 1):
        cols_to_drop.extend(FEATURE_GROUPS.get(g, []))

    # only drop those that actually exist
    cols_to_drop = [c for c in cols_to_drop if c in X_df.columns]
    return X_df.drop(columns=cols_to_drop)

In [194]:
def features_none(df):
    return df

def features_simple(df):
    ltm = LaggedTargetMomentum(
        target_col="lagged_target",
        use_log=False,
        drop_source=False,
    )
    return ltm.fit_transform(df)

def features_medium(df):
    fb = FeatureBuilder(
        windows=(10, 21, 63),
        lags=(1, 5, 21),
        excluded_cols=EXCLUDED_COLS,
        verbose=False,
    )
    df_ext = fb.fit_transform(df)
    ltm = LaggedTargetMomentum(
        target_col="lagged_target",
        use_log=False,
        drop_source=False
    )
    return ltm.fit_transform(df_ext)

def features_extensive(df):
    fb = FeatureBuilder(
        windows=(5,10,21,63,126,252),
        lags=(1,2,3,5,10,21,63,126),
        excluded_cols=EXCLUDED_COLS,
        verbose=False,
    )
    df_ext = fb.fit_transform(df)
    ltm = LaggedTargetMomentum(
        target_col="lagged_target",
        use_log=False,
        drop_source=False
    )
    df_ext = ltm.fit_transform(df)
    vi = VolatilityIndicators()
    return vi.fit_transform(df_ext)


FEATURE_CONDITIONS = {
    "none": features_none,
    "simple": features_simple,
    "medium": features_medium,
    "extensive": features_extensive
}

In [195]:
from tqdm.auto import tqdm

def _get_best_iteration(xgb_est):
    bi = getattr(xgb_est, "best_iteration", None)
    if bi is None:
        try:
            bi = xgb_est.get_booster().best_iteration
        except Exception:
            bi = None
    return bi

LEAK_PREFIXES = [
    "forward_returns",
    "market_forward_excess_returns",
    "risk_free_rate",
    # "lagged_forward_returns",
    # "lagged_risk_free_rate",
    # "lagged_market_forward_excess_returns",
]

def _drop_leakage(df_t):
    if hasattr(df_t, "columns"):
        cols = df_t.columns
        to_drop = [c for c in cols if any(c.startswith(p) for p in LEAK_PREFIXES)]
        if to_drop:
            print("[DEBUG] Dropping leakage columns from transformed features:")
            for c in to_drop:
                print("   ", c)
            df_t = df_t.drop(columns=to_drop)
    return df_t



def walk_forward_cv_with_positions(
    pipe,
    df, 
    X, y, 
    row_id_col = "row_id",
    n_splits = 2,
    test_size = 252,
    gap = 1,
    early_stopping_rounds = 100,
    eval_tail_max = 252,
    eval_tail_min = 50,
    vol_col = "MOM_y_std_21",
    regime_col = "VI_regime_highvol",
    position_fn = None,
    sharpe_fn = None
):
    tscv = TimeSeriesSplit(n_splits=n_splits, test_size=test_size)
    fold_metrics = []
    oof_rows = []
    best_iters = []
    rmse_list, r2_list, mse_list, mae_list, adj_sharpes = [], [], [], [], []

    print(f"Starting Walk-Forward Validation with {n_splits} folds (Test size: {test_size} days)")
    print("\n" + ("-" * 50) + "\n")
            
    for fold, (train_index, test_index) in enumerate(tscv.split(X), 1):
        # one progress bar per fold
        steps = [
            "split train/test",
            "build eval tail",
            "fit preprocessor",
            "transform data",
            "fit model",
            "predict",
            "compute metrics",
            "build positions",
            "compute Sharpe",
            "accumulate results",
        ]
        pbar = tqdm(
            total=len(steps),
            desc=f"Fold {fold}: starting",
            leave=False,
            bar_format="{l_bar}{bar} {n_fmt}/{total_fmt}"
        )

        # 1) split train/test (with gap)
        pbar.set_description(f"Fold {fold}: {steps[0]}")
        if gap > 0 and len(test_index) > 0:
            max_train_index = test_index[0] - gap
            train_index = train_index[train_index < max_train_index]
            
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]
        pbar.update(1)

        # 2) build eval tail
        pbar.set_description(f"Fold {fold}: {steps[1]}")
        eval_tail = min(len(X_train)//5, eval_tail_max)
        eval_tail = max(eval_tail, eval_tail_min)

        X_train_core = X_train.iloc[:-eval_tail]
        y_train_core = y_train.iloc[:-eval_tail]
        X_ev = X_train.iloc[-eval_tail:]
        y_ev = y_train.iloc[-eval_tail:]
        pbar.update(1)

        # 3) clone pipeline & fit preprocessor
        pbar.set_description(f"Fold {fold}: {steps[2]}")
        pipe_fold = clone(pipe)
        preprocessor = Pipeline(pipe_fold.steps[:-1])
        base_model = pipe_fold.steps[-1][1]   # the XGBRegressor

        preprocessor.fit(X_train, y_train)
        pbar.update(1)

        # 4) transform data
        pbar.set_description(f"Fold {fold}: {steps[3]}")
        X_train_t = preprocessor.transform(X_train)
        X_train_core_t = preprocessor.transform(X_train_core)
        X_ev_t = preprocessor.transform(X_ev)
        X_test_t = preprocessor.transform(X_test)

        X_train_t = _drop_leakage(X_train_t)
        X_train_core_t = _drop_leakage(X_train_core_t)
        X_ev_t = _drop_leakage(X_ev_t)
        X_test_t = _drop_leakage(X_test_t)
        
        if fold == 1:   # or any fold you want to inspect
            # If your transformers return DataFrames (they should), this works:
            if hasattr(X_train_t, "columns"):
                feature_cols = list(X_train_t.columns)
            else:
                # fallback if some step returns numpy: try to ask the last step
                align_step = None
                for name, step in preprocessor.steps[::-1]:
                    if hasattr(step, "output_columns_"):
                        align_step = step
                        break
                if align_step is not None:
                    feature_cols = list(align_step.output_columns_)
                else:
                    feature_cols = [f"f{i}" for i in range(X_train_t.shape[1])]

            # print(f"\n[DEBUG] Fold {fold}: {len(feature_cols)} features going into model\n")
            # print("[DEBUG] First 40 columns:")
            # print(feature_cols[:40])

            # # look explicitly for things that should NOT be there
            # leak_keywords = [
            #     "forward_returns",
            #     "market_forward_excess_returns",
            #     "log_market_forward_excess_returns",
            #     "risk_free",
            #     "lagged_forward_returns",
            #     "lagged_market_forward_excess_returns",
            #     "target",
            #     "y_",
            # ]
            # leak_cols = [c for c in feature_cols if any(k in c for k in leak_keywords)]
            # print("\n[DEBUG] Potential leakage columns:")
            # for c in leak_cols:
            #     print("   ", c)

            # # optional: keep a copy for manual poking later
            # global DEBUG_FEATURE_DF  # yes, this is ugly, you’re debugging
            # if hasattr(X_train_t, "columns"):
            #     DEBUG_FEATURE_DF = X_train_t.copy()
            # else:
            #     DEBUG_FEATURE_DF = pd.DataFrame(X_train_t, columns=feature_cols)

            # print("\n[DEBUG] Stored first fold feature matrix in DEBUG_FEATURE_DF\n")

        
        # then convert to numpy
        X_train_t = np.asarray(X_train_t)
        X_train_core_t = np.asarray(X_train_core_t)
        X_ev_t = np.asarray(X_ev_t)
        X_test_t = np.asarray(X_test_t)
                
        X_train_t = np.asarray(X_train_t)
        X_train_core_t = np.asarray(X_train_core_t)
        X_ev_t = np.asarray(X_ev_t)
        X_test_t = np.asarray(X_test_t)
        pbar.update(1)
        
        # 5) fit model (with early stopping)
        pbar.set_description(f"Fold {fold}: {steps[4]}")
        model = clone(base_model)
        model.set_params(
            eval_metric="rmse",
            early_stopping_rounds=early_stopping_rounds,
        )
        model.fit(
            X_train_t,
            y_train,
            eval_set=[(X_train_core_t, y_train_core), (X_ev_t, y_ev)],
            verbose=False,
        )
        pbar.update(1)

        # 6) predict
        pbar.set_description(f"Fold {fold}: {steps[5]}")
        y_hat_train = model.predict(X_train_t)
        y_hat_test = model.predict(X_test_t)
        pbar.update(1)

        # 7) compute metrics
        pbar.set_description(f"Fold {fold}: {steps[6]}")
        fold_r2 = r2_score(y_test, y_hat_test)
        fold_mse = mean_squared_error(y_test, y_hat_test)
        fold_rmse = mean_squared_error(y_test, y_hat_test, squared=False)
        fold_mae = mean_absolute_error(y_test, y_hat_test)
        rmse_list.append(fold_rmse)
        r2_list.append(fold_r2)
        mae_list.append(fold_mae)
        mse_list.append(fold_mse)
        pbar.update(1)

        # 8) build positions
        pbar.set_description(f"Fold {fold}: {steps[7]}")
        if vol_col in df.columns:
            vol_series = df.loc[X_test.index, vol_col]
            vol_factor_test = (vol_series.astype(float) /
                               float(df[vol_col].median()))
        else:
            vol_factor_test = np.ones(len(X_test))

        train_resid = (y_train - y_hat_train)
        rmse_tail = float(np.sqrt(train_resid.pow(2).rolling(126, min_periods=50).mean().iloc[-1]))
        rmse_base = float(np.sqrt(train_resid.pow(2).rolling(252, min_periods=50).mean()).median())
        if not np.isfinite(rmse_base) or rmse_base == 0:
            rmse_base = max(rmse_tail, 1e-6)
        rmse_factor_test = np.full(len(X_test), rmse_tail / rmse_base, dtype=float)

        regime_test = df.loc[X_test.index, regime_col].to_numpy() if regime_col in df.columns else np.zeros(len(X_test))

        if position_fn is None:
            pos_te = y_hat_test.copy()
        else:
            pos_te = position_fn(
                y_hat_train=y_hat_train,
                y_hat_test=y_hat_test,
                vol_factor_test=vol_factor_test,
                rmse_factor_test=rmse_factor_test,
                regime_test=regime_test
            ).astype(float)
        pbar.update(1)

        # 9) compute Sharpe
        pbar.set_description(f"Fold {fold}: {steps[8]}")
        if sharpe_fn is None:
            fold_adj_sharpe = np.nan
        else:
            solution_fold = df.loc[X_test.index, [row_id_col, "forward_returns", "risk_free_rate"]].copy()
            submission_fold = pd.DataFrame({
                row_id_col: df.loc[X_test.index, row_id_col].values,
                "prediction": pos_te
            })
            fold_adj_sharpe = float(sharpe_fn(solution_fold, submission_fold, row_id_col=row_id_col))
        adj_sharpes.append(fold_adj_sharpe)
        pbar.update(1)

        # 10) accumulate results
        pbar.set_description(f"Fold {fold}: {steps[9]}")
        oof_rows.append(pd.DataFrame({
            "y_log": y_test.values,
            "y_arith": np.expm1(y_test.values),
            "y_pred_log": y_hat_test,
            "y_pred_arith": np.expm1(y_hat_test),
            "position": pos_te,
            "fold": fold
        }))

        bi = _get_best_iteration(model)
        if bi is not None:
            best_iters.append(int(bi))
        pbar.update(1)

        # kill the bar for this fold before printing summary
        pbar.close()

        print(
            f"Fold {fold}: n_tr={len(X_train_t):,}, n_te={len(X_test):,}  "
            f"MSE={fold_mse:.5f}  RMSE={fold_rmse:.5f}  R2={fold_r2:.4f}  "
            f"MAE={fold_mae:.5f}  AdjSharpe={fold_adj_sharpe}"
        )

    # aggregate
    oof = (
        pd.concat(oof_rows, ignore_index=True)
        .sort_values("fold")
        .reset_index(drop=True)
    )
    metrics_df = pd.DataFrame({
        "fold": np.arange(1, len(rmse_list)+1),
        "rmse": rmse_list,
        "mse": mse_list,
        "mae": mae_list,
        "r2": r2_list,
        "adjusted_sharpe": adj_sharpes + [np.nan]*(len(rmse_list)-len(adj_sharpes))
    })

    results = {
        "fold_metrics": metrics_df,
        "oof_rows": oof,
        "best_iters": best_iters,
        "rmse_mean": float(np.mean(rmse_list)), "rmse_std": float(np.std(rmse_list)),
        "r2_mean": float(np.mean(r2_list)), "r2_std": float(np.std(r2_list)),
        "mae_mean": float(np.mean(mae_list)), "mae_std": float(np.std(mae_list)),
        "adj_sharpe_mean": float(np.nanmean(metrics_df["adjusted_sharpe"])) if len(adj_sharpes) else np.nan,
        "adj_sharpe_max": float(np.nanmax(metrics_df["adjusted_sharpe"])) if len(adj_sharpes) else np.nan,
    }
    return results


In [196]:
def build_x_y_for_condition(df_raw, impute_key, drop_level, feat_key, drop_features_bool):
    # Base target + imputation
    df0, y = make_base_target(df_raw)
    df_imp = IMPUTERS[impute_key](df0)

    # Optionally drop feature groups BEFORE feature construction
    if drop_features_bool:
        df_base = drop_feature_groups(df_imp, drop_level)
    else:
        df_base = df_imp

    # Apply feature transformation
    df_feat = FEATURE_CONDITIONS[feat_key](df_base)

    # Build X from feature DataFrame, excluding unwanted columns
    X = df_feat.drop(columns=EXCLUDED_COLS, errors="ignore")

    # 1) Align X and y on a common index
    X, y = X.align(y, join="inner", axis=0)

    # 2) Clean infs -> NaNs in X
    X = X.replace([np.inf, -np.inf], np.nan)

    # 3) Drop any rows in X that still contain NaNs
    mask = X.notna().all(axis=1)
    X = X.loc[mask]
    y = y.loc[mask]

    # 4) Keep df_feat in sync with X
    df_feat = df_feat.loc[X.index]

    return df_feat, X, y

    
# def build_x_y_for_condition(df_raw, impute_key, drop_level, feat_key, corr_key, impor_key):
#     df0, y = make_base_target(df_raw)
    
#     df_imp = IMPUTERS[impute_key](df0)
    
#     df_drop = drop_feature_groups(df_imp, drop_level)
#     df_feat = FEATURE_CONDITIONS[feat_key](df_drop)

#     X = df_feat.drop(columns=EXCLUDED_COLS, errors="ignore")
#     top_corr = getTopCorrColumns(df, corr_key)
#     top_impor = getTopImportanceColumns(df, impor_key)
#     cols_keep = top_corr[]
#     X = X.drop
    
#     # 1) First, align X and y on a common index
#     common_idx = X.index.intersection(y.index)
#     X = X.loc[common_idx]
#     y = y.loc[common_idx]

#     # 2) Clean infs -> NaNs in X
#     X = X.replace([np.inf, -np.inf], np.nan)

#     # 3) Build mask from X only
#     mask = X.notna().all(axis=1)   # mask has index = X.index

#     # 4) Use that mask's *index* to slice both X and y
#     valid_idx = X.index[mask]

#     X = X.loc[valid_idx]
#     y = y.loc[valid_idx]

#     # 5) Optionally reduce df_feat to the same rows
#     df_feat = df_feat.loc[valid_idx]

#     return df_feat, X, y

In [197]:
def make_ols():
    return Pipeline([
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ])

def make_ridge(alpha=0.1):
    return Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=alpha, random_state=42))
    ])

def make_lasso(alpha=1e-3):
    return Pipeline([
        ("scaler", StandardScaler()),
        ("model", Lasso(alpha=alpha, random_state=42, max_iter=10000))
    ])

def make_elasticnet(alpha=1e-3, l1_ratio=0.5):
    return Pipeline([
        ("scaler", StandardScaler()),
        ("model", ElasticNet(alpha=alpha, l1_ratio=l1_ratio, random_state=42, max_iter=10000))
    ])

def make_lgbm(params=None):
    params = params or {}
    base = dict(
        objective="regression",
        metric="rmse",
        boosting_type="gbdt",
        n_estimators=500,
        learning_rate=42,
    )
    base.update(params)
    return LGBMRegressor(**base)

def make_xgb(params=None):
    params = params or {}
    base = dict(
        objective="reg:squarederror",
        n_estimators=600,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method="hist",
        random_state=42,
        device="cuda"
    )
    base.update(params)
    return XGBRegressor(**base)

In [198]:
def eval_standard_cv(model, X, y, n_splits=10):
    tscv=TimeSeriesSplit(n_splits=n_splits)
    scores_rmse = cross_val_score(
        model,
        X, y,
        cv=tscv,
        scoring="neg_root_mean_squared_error",
    )
    scores_r2 = cross_val_score(
        model,
        X, y,
        cv=tscv,
        scoring="r2",
    )
    return {
        "rmse_mean": -scores_rmse.mean(),
        "rmse_std": scores_rmse.std(),
        "r2_mean": scores_r2.mean(),
        "r2_std": scores_r2.std()
    }

In [199]:
def xgb_from_trial(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 1200),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 10.0),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "tree_method": "hist",
        "objective": "reg:squarederror",
        "n_jobs": -1,
        "random_state": 42,
    }
    return XGBRegressor(**params)

def optuna_objective_tree(trial, X, y, df_eval):
    xgb_model = xgb_from_trial(trial)
    pipe = base_pipeline(xgb_model)   # your feature builder + target momentum + guard/align

    results = walk_forward_cv_with_positions(
        pipe=pipe,
        df=df_eval,
        X=X,
        y=y,
        row_id_col=ROW_ID_COL,
        n_splits=N_SPLITS,
        test_size=TEST_SIZE,
        gap=GAP,
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        vol_col="MOM_y_std_21",
        regime_col="VI_regime_highvol",
        position_fn=signal_to_position,
        sharpe_fn=adjusted_sharpe_from_df,
    )

    adj_sharpe_mean = results.get("adj_sharpe_mean", float("nan"))
    if not np.isfinite(adj_sharpe_mean):
        raise optuna.exceptions.TrialPruned()

    # log some extras
    trial.set_user_attr("rmse_mean", results["rmse_mean"])
    trial.set_user_attr("r2_mean", results["r2_mean"])

    return adj_sharpe_mean

In [200]:
def tune_xgb_with_optuna(X, y, df_eval, n_trials=30):
    study = optuna.create_study(
        study_name="xgb_wf_sharpe",
        direction="maximize",
    )
    study.optimize(
        lambda trial: optuna_objective_tree(trial, X, y, df_eval),
        n_trials=n_trials,
        show_progress_bar=True,
    )
    return study

In [201]:
def get_tuning_slice(X, y, frac=0.8):
    """
    Use the first `frac` fraction of the sample for inner CV tuning.
    Assumes X and y are already sorted in time order.
    """
    n = len(X)
    cutoff = int(n * frac)
    return X.iloc[:cutoff], y.iloc[:cutoff]

def _scale_for_tuning(X):
    scaler = StandardScaler()
    return scaler.fit_transform(X)

In [202]:
def tune_ridge_alpha_for_config(X, y, frac=0.8, n_splits=5):
    """
    Time-series CV to get best Ridge alpha for a given (impute, feat) config.
    """
    X_inner, y_inner = get_tuning_slice(X, y, frac=frac)

    X_inner_scaled = _scale_for_tuning(X_inner)

    alphas = np.logspace(-4, 3, 15)
    tscv = TimeSeriesSplit(n_splits=n_splits)

    ridge_cv = RidgeCV(
        alphas=alphas,
        cv=tscv,
        scoring="neg_mean_squared_error",
    )
    ridge_cv.fit(X_inner_scaled, y_inner)
    return float(ridge_cv.alpha_)

def tune_lasso_alpha_for_config(X, y, frac=0.8, n_splits=5):
    """
    Time-series CV to get best Lasso alpha for a given (impute, feat) config.
    """
    X_inner, y_inner = get_tuning_slice(X, y, frac=frac)
    
    X_inner_scaled = _scale_for_tuning(X_inner)

    alphas = np.logspace(-3, 0.5, 10)
    tscv = TimeSeriesSplit(n_splits=n_splits)

    lasso_cv = LassoCV(
        alphas=alphas,
        cv=tscv,
        random_state=42,
        max_iter=50000,
    )
    lasso_cv.fit(X_inner_scaled, y_inner)
    return float(lasso_cv.alpha_)

def tune_elasticnet_for_config(X, y, frac=0.8, n_splits=5):
    """
    Time-series CV to get best ElasticNet alpha and l1_ratio.
    """
    X_inner, y_inner = get_tuning_slice(X, y, frac=frac)

    X_inner_scaled = _scale_for_tuning(X_inner)

    alphas = np.logspace(-3, 0.5, 10)
    l1_ratios = [0.1, 0.3, 0.5, 0.7, 0.9]
    tscv = TimeSeriesSplit(n_splits=n_splits)

    enet_cv = ElasticNetCV(
        alphas=alphas,
        l1_ratio=l1_ratios,
        cv=tscv,
        random_state=42,
        max_iter=50000,
    )
    enet_cv.fit(X_inner_scaled, y_inner)
    best_alpha = float(enet_cv.alpha_)
    best_l1 = float(enet_cv.l1_ratio_)
    return {"alpha": best_alpha, "l1_ratio": best_l1}

def tune_lgbm_with_gridsearch(X, y, frac=0.8, n_splits=3):
    X_inner, y_inner = get_tuning_slice(X, y, frac=frac)

    model = LGBMRegressor(
        objective="regression",
        random_state=42,
        n_jobs=1,
        verbose=-1,
        device="cuda"
    )

    param_grid = {
        "num_leaves":        [16],
        "max_depth":         [8],
        "learning_rate":     [0.05],
        "min_child_samples": [25],
        "subsample":         [0.8],
        "colsample_bytree":  [0.8],
        "n_estimators":      [8000]
    }

    tscv = TimeSeriesSplit(n_splits=n_splits)

    grid = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        scoring="neg_mean_squared_error",
        cv=tscv,
        n_jobs=4,
        verbose=1
    )

    grid.fit(X_inner, y_inner)
    
    best_params = grid.best_params_
    return best_params

def tune_xgb_with_gridsearch(X, y, frac=0.8, n_splits=3):
    X_inner, y_inner = get_tuning_slice(X, y, frac=frac)
    X_inner_gpu = cp.asarray(X_inner.values)
    if n_splits == 1:
        params = {
            # 'colsample_bylevel': 0.8,
            # 'colsample_bynode': 0.3,

            'max_depth': 7,
            'min_child_weight': 3,
            'subsample': 0.8,
            'colsample_bytree': 0.6,
            'learning_rate': 0.005,
            
            # 'max_leaves': 256,

            'n_estimators': 5000,
            # 'reg_alpha': 0.2,
            # 'reg_lambda': 3.25,

        }
        return params
    else:
        model = XGBRegressor(
            objective="reg:squarederror",
            tree_method="hist",
            random_state=42,
            n_jobs=1,
            device="cuda",
        )
    
        param_grid = {
            "max_depth": [7],
            "min_child_weight": [4],
            "subsample": [0.7],
            "colsample_bytree": [0.6],
            "learning_rate": [0.005],
            "n_estimators": [4000],
            
            # just do these probably.
            # "max_depth":        [6], #probably too low
            # "n_estimators":     [5000], #too low
            # "learning_rate":    [0.1],
            # "min_child_weight": [2], # 
            
            # "subsample":        [0.6], # try 0.6
            # "colsample_bytree": [0.6], #
            # "colsample_bylevel":[0.8],
            "max_leaves":       [32, 64, 128], # Tiny try 256
    
            # "reg_lambda":       [3.25], # l2 3.25
            # "reg_alpha":        [0.2], # l1 0.2
            # "colsample_bynode": [0.3]
        }
    
        tscv = TimeSeriesSplit(n_splits=n_splits)
    
        grid = GridSearchCV(
            estimator=model,
            param_grid=param_grid,
            scoring="neg_mean_squared_error",
            cv=tscv,
            n_jobs=1,
            verbose=4
        )
    
        grid.fit(X_inner_gpu, y_inner)
        best_params = grid.best_params_
        print(best_params)
        return best_params

In [203]:
def train_test_split_time(X, y, test_frac=0.2):
    """
    Split X, y into (train, test) by time order.

    Assumes X and y are already sorted in time (which they are,
    because make_base_target sorts by DATE_COL before everything else).
    """
    n = len(X)
    split_idx = int(n * (1 - test_frac))
    X_train = X.iloc[:split_idx]
    X_test  = X.iloc[split_idx:]
    y_train = y.iloc[:split_idx]
    y_test  = y.iloc[split_idx:]
    return X_train, X_test, y_train, y_test

In [204]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def eval_holdout(model, X_train, y_train, X_test, y_test):
    """
    Fit model on (X_train, y_train) and evaluate on (X_test, y_test).
    Returns a dict with RMSE/MAE/R2 for the final holdout segment.
    """
    model.fit(X_train, y_train)
    insample_y_pred = model.predict(X_train)
    holdout_y_pred = model.predict(X_test)

    # insample_rmse = np.sqrt(mean_squared_error(y_train, insample_y_pred))
    # insample_mae  = mean_absolute_error(y_train, insample_y_pred)
    insample_r2   = r2_score(y_train, insample_y_pred)

    # holdout_rmse = np.sqrt(mean_squared_error(y_test, holdout_y_pred))
    # holdout_mae  = mean_absolute_error(y_test, holdout_y_pred)
    holdout_r2   = r2_score(y_test, holdout_y_pred)
    
    return {
        # "insample_rmse": np.sqrt(mean_squared_error(y_train, insample_y_pred)),
        # "insample_mae": mean_absolute_error(y_train, insample_y_pred),
        "r2_insample": insample_r2,
        # "rmse_holdout": holdout_rmse,
        # "mae_holdout": holdout_mae,
        "r2_holdout": holdout_r2,
    }

In [205]:
# def findTopCorrFeaturesElastic(X_train, y_train, best_enet, k=20):
#     pipe = make_elasticnet(alpha=best_enet["alpha"], l1_ratio=best_enet["l1_ratio"])
#     pipe.fit(X_train, y_train)
#     coefs = pipe.named_steps["model"].coef_
#     feature_names = np.array(X_train.columns)

#     # 3) get top-k by |coef|
#     idx_sorted = np.argsort(np.abs(coefs))[::-1]
#     idx_top = idx_sorted[:k]
#     top_features = feature_names[idx_top]

#     return list(top_features)

# def findTopImporFeaturesXGB(X_train, y_train, best_xgb, k=20):
#     XGB = make_xgb(params=best_xgb)
#     XGB.fit(X_train, y_train)
#     importances = XGB.feature_importances_
#     feature_names = np.array(X_train.columns)

#     # Sort by importance descending
#     idx_sorted = np.argsort(importances)[::-1]
#     idx_top = idx_sorted[:k]

#     top_features = feature_names[idx_top]
#     return list(top_features)

# results_table = []

# # impute_options = ["medium", "advanced"]
# drop_levels    = [3]
# feat_options   = ["medium",]
# impute_options = ["medium"]
# # drop_levels    = [3]
# # feat_options   = ["none"]
# correlation_windows = [50]
# feature_importance_windows = [50]
# drop_features = [False]

# configs = list(product(impute_options, drop_levels, feat_options, correlation_windows, feature_importance_windows, drop_features))
# # configs = list(product(impute_options, drop_levels, feat_options))

# # for impute_key, drop_level, feat_key, corr_key, impor_key in tqdm(configs, desc="Configs", position=0, leave=True):
# #     print(f"Building X y with conditions -> impute: {impute_key} drop: {drop_level} features: {feat_key}")
# #     df_cond, X_cond, y_cond = build_x_y_for_condition(df_raw, impute_key, drop_level, feat_key, corr_key, impor_key)
# import cupy as cp

# X_train_gpu = cp.asarray(X_train.values)
# X_test_gpu  = cp.asarray(X_test.values)

# for impute_key, drop_level, feat_key, corr_key, impor_key, drop_features_bool in tqdm(configs, desc="Configs", position=0, leave=True):
#     print(f"Building X y with conditions -> impute: {impute_key} drop: {drop_level} features: {feat_key}, drop_features: {drop_features_bool}")
#     df_cond, X_cond, y_cond = build_x_y_for_condition(df_raw, impute_key, drop_level, feat_key, drop_features_bool)
#     # Filter function - absolute correlation and importance top (20, 30, 40) 
#     # Create Linear X_cond based on feature importance, correlation Shap
    
#     X_train, X_test, y_train, y_test = train_test_split_time(X_cond, y_cond, test_frac=0.2)
    
#     # print(f"Running Ridge CV")
#     # best_alpha_ridge = tune_ridge_alpha_for_config(X_cond, y_cond, frac=0.8, n_splits=5)
#     # print(f"Running lasso CV")
#     # best_alpha_lasso = tune_lasso_alpha_for_config(X_cond, y_cond, frac=0.8, n_splits=5)
#     # print(f"Running ElsticNet CV")
#     # best_enet = tune_elasticnet_for_config(X_cond, y_cond)

#     if drop_features_bool == True:
#         elasticnet_top_features = findTopCorrFeaturesElastic(X_train, y_train, best_enet, corr_key)
#         X_train_red = X_train[elasticnet_top_features].copy()
#         X_test_red  = X_test[elasticnet_top_features].copy()

#     X_train_gpu = cp.asarray(X_train.values)
#     X_test_gpu  = cp.asarray(X_test.values)
#     # print(f"Running LGBM GridSearch CV")
#     # best_lgbm = tune_lgbm_with_gridsearch(X_train, y_train, frac=0.8, n_splits=2)
#     print(f"Running XGBoost GridSearch CV")
#     best_xgb = tune_xgb_with_gridsearch(X_train_gpu, y_train, frac=0.8, n_splits=5)



#     if drop_features_bool == True:
#         xgb_top_features = findTopImporFeaturesXGB(X_train, y_train, best_xgb, impor_key)
#         X_train_blue = X_train[xgb_top_features].copy()
#         X_test_blue = X_test[xgb_top_features].copy()
        
#     print("Initialising Model Configs")
#     models_config = {
#         # "ols": make_ols(),
#         # "ridge": make_ridge(alpha=best_alpha_ridge),
#         # "lasso":      make_lasso(alpha=best_alpha_lasso),
#         # "elasticnet": make_elasticnet(alpha=best_enet["alpha"], l1_ratio=best_enet["l1_ratio"]),
#         # "lgbm":       make_lgbm(params=best_lgbm),
#         "xgb":        make_xgb(params=best_xgb)
#     }
#     for model_name, model in tqdm(
#         models_config.items(),
#         desc=f"Models ({impute_key}, drop={drop_level}, feat={feat_key})",
#         leave=False
#     ):
        
#         if (drop_features_bool == True):
#             X_train_eval = []
#             X_test_eval = []
#             if (model_name == "elasticnet" or model_name == "lasso"):
#                 X_train_eval = X_train_red
#                 X_test_eval = X_test_red
#             else:
#                 X_train_eval = X_train_blue
#                 X_test_eval = X_test_blue
    
#             # Standard CV
#             print(f"Evaluating model {model_name}")
#             holdout_metrics = eval_holdout(model, X_train_eval, y_train, X_test_eval, y_test)
#         else:
#             print(f"Evaluating model {model_name}")
#             holdout_metrics = eval_holdout(model, X_train_gpu, y_train, X_test_gpu, y_test)
#         print(holdout_metrics)
#         results_table.append({
#             "model": model_name,
#             "impute": impute_key,
#             "drop_level": drop_level,
#             "features": feat_key,
#             "corr_key": corr_key,
#             "impor_key": impor_key,
#             "drop_features": drop_features_bool,
#             "mode": "standard_cv",
#             # "best_alpha_ridge": best_alpha_ridge,
#             # "best_alpha_lasso": best_alpha_lasso,
#             # "best_enet": best_enet,
#             # "best_lgbm": best_lgbm,
#             "best_xgb": best_xgb,
#             # **std_metrics,
#             **holdout_metrics
#         })

#         # # Optuna Sharpe tuning only for tree models (or only on best few combos)
#         # if model_name in ("xgb", "lgbm") and feat_key != "none":
#         #     study = tune_xgb_with_optuna(X_cond, y_cond, df_cond, n_trials=20)
#         #     best_val = study.best_value
#         #     results_table.append({
#         #         "model": model_name,
#         #         "impute": impute_key,
#         #         "features": feat_key,
#         #         "mode": "wf_optuna_sharpe",
#         #         "adj_sharpe_best": best_val,
#         #     })

In [206]:
# import pandas as pd

# IMPUTE_ORDER = ["medium", "advanced"]
# DROP_GROUPS  = [[3]]
# FEAT_ORDER   = ["none", "simple","medium", "extensive"]

# def wrap_num(x):
#     if pd.isna(x):
#         return ""
#     return rf"\num{{{x}}}"

# def make_std_table(df_std, metric: str) -> str:
#     """
#     Create one LaTeX table (with all imputations and drop-level groups)
#     for a given metric.
#     """
#     pivot = df_std.pivot_table(
#         index="model",
#         columns=["impute", "drop_level", "features"],
#         values=metric,
#         aggfunc="mean",
#     )

#     all_drop_levels = sorted(df_std["drop_level"].unique())
#     full_cols = pd.MultiIndex.from_product(
#         [IMPUTE_ORDER, all_drop_levels, FEAT_ORDER],
#         names=pivot.columns.names,
#     )
#     pivot = pivot.reindex(columns=full_cols)

#     lines = []
#     metric_tex = metric.upper().replace("_", r"\_")

#     lines.append(r"\begin{table}")
#     lines.append(r"\centering")
#     lines.append(
#         rf"\caption{{Standard-CV {metric_tex} by model, imputation, drop level, and feature set}}"
#     )
#     lines.append(rf"\label{{tab:model_impute_drop_features_{metric}}}")
#     lines.append(r"\small")

#     for imp in IMPUTE_ORDER:
#         lines.append("")
#         lines.append(rf"\textbf{{Imputation: {imp}}}\\[0.3em]")

#         sub = pivot[imp]  # columns: (drop_level, features)

#         for grp in DROP_GROUPS:
#             group_cols = pd.MultiIndex.from_product(
#                 [grp, FEAT_ORDER], names=sub.columns.names
#             )
#             sub_g = sub.reindex(columns=group_cols)

#             n_d = len(grp)
#             lines.append(rf"\begin{{tabular}}{{l*{{{n_d}}}{{rrrr}}}}")
#             lines.append(r"\toprule")

#             # header 1: drop_level
#             header1 = "drop_level & " + " & ".join(
#                 rf"\multicolumn{{4}}{{c}}{{{dl}}}" for dl in grp
#             ) + r" \\"
#             lines.append(header1)

#             # header 2: features
#             header2 = (
#                 "features & "
#                 + " & ".join(FEAT_ORDER * n_d)
#                 + r" \\"
#             )
#             lines.append(header2)
#             lines.append(r"\midrule")

#             # rows (sorted models for stability)
#             for model in sorted(sub_g.index):
#                 vals = []
#                 for dl in grp:
#                     for feat in FEAT_ORDER:
#                         vals.append(wrap_num(sub_g.loc[model, (dl, feat)]))
#                 row = model + " & " + " & ".join(vals) + r" \\"
#                 lines.append(row)

#             lines.append(r"\bottomrule")
#             lines.append(r"\end{tabular}")
#             lines.append(r"\vspace{0.5em}")

#         lines.append(r"\vspace{1em}")

#     lines.append(r"\end{table}")
#     return "\n".join(lines)


# DOC_TEMPLATE = r"""
# \documentclass{article}
# \usepackage{graphicx} % Required for inserting images
# \usepackage{rotating}
# \usepackage{pdflscape}
# \usepackage[margin=0.5cm]{geometry} % smaller margins
# \setlength{\tabcolsep}{1pt}      % tighter columns
# \usepackage{booktabs} % for \toprule etc
# \usepackage{siunitx}
# \sisetup{
#   round-mode = places,
#   round-precision = 4,
#   exponent-mode = threshold,
#   exponent-thresholds = {-5:4} % decimals for 10^{-4}..10^{4}, sci outside
# }
# \begin{document}

# \title{Kaggle}
# \author{Lewis Hikari Kawase Kennedy}
# \date{November 2025}

# \maketitle

# \section{Introduction}
# \clearpage
# \begin{landscape}
# *TABLES GOES HERE*
# \end{landscape}

# \end{document}
# """

# METRIC_COLS = ["r2_insample"]
# OOF_COLS = ["r2_holdout"]

# COLS = ["r2_insample", "r2_holdout"]

# def make_full_document(df_std) -> str:
#     table_blocks = []
#     # for col in METRIC_COLS:
#     #     table_blocks.append(make_std_table(df_std, col))
#     # for col in OOF_COLS:
#     #     table_blocks.append(make_std_table(df_std, col))
#     for col in COLS:
#         table_blocks.append(make_std_table(df_std, col))
#     all_tables = "\n\n".join(table_blocks)
#     return DOC_TEMPLATE.replace("*TABLES GOES HERE*", all_tables)


# # Example usage:
# df_res = pd.DataFrame(results_table)
# df_std = df_res[df_res["mode"] == "standard_cv"].copy()
# tex_source = make_full_document(df_res)
# print(tex_source)
# with open("kaggle_tables.tex", "w") as f:
#     f.write(tex_source)

# print(df_res)

In [207]:
def findTopCorrFeaturesElastic(X_train, y_train, best_enet, k=20):
    pipe = make_elasticnet(alpha=best_enet["alpha"], l1_ratio=best_enet["l1_ratio"])
    pipe.fit(X_train, y_train)
    coefs = pipe.named_steps["model"].coef_
    feature_names = np.array(X_train.columns)

    # 3) get top-k by |coef|
    idx_sorted = np.argsort(np.abs(coefs))[::-1]
    idx_top = idx_sorted[:k]
    top_features = feature_names[idx_top]

    return list(top_features)

def findTopImporFeaturesXGB(X_train, y_train, best_xgb, k=20):
    XGB = make_xgb(params=best_xgb)
    XGB.fit(X_train, y_train)
    importances = XGB.feature_importances_
    feature_names = np.array(X_train.columns)

    # Sort by importance descending
    idx_sorted = np.argsort(importances)[::-1]
    idx_top = idx_sorted[:k]

    top_features = feature_names[idx_top]
    return list(top_features)

results_table = []

# impute_options = ["medium", "advanced"]
drop_levels    = [3]
feat_options   = ["medium",]
impute_options = ["medium"]
# drop_levels    = [3]
# feat_options   = ["none"]
correlation_windows = [50]
feature_importance_windows = [50]
drop_features = [False]

configs = list(product(impute_options, drop_levels, feat_options, correlation_windows, feature_importance_windows, drop_features))
# configs = list(product(impute_options, drop_levels, feat_options))

# for impute_key, drop_level, feat_key, corr_key, impor_key in tqdm(configs, desc="Configs", position=0, leave=True):
#     print(f"Building X y with conditions -> impute: {impute_key} drop: {drop_level} features: {feat_key}")
#     df_cond, X_cond, y_cond = build_x_y_for_condition(df_raw, impute_key, drop_level, feat_key, corr_key, impor_key)
import cupy as cp


for impute_key, drop_level, feat_key, corr_key, impor_key, drop_features_bool in tqdm(configs, desc="Configs", position=0, leave=True):
    print(f"Building X y with conditions -> impute: {impute_key} drop: {drop_level} features: {feat_key}, drop_features: {drop_features_bool}")
    df_cond, X_cond, y_cond = build_x_y_for_condition(df_raw, impute_key, drop_level, feat_key, drop_features_bool)
    # Filter function - absolute correlation and importance top (20, 30, 40) 
    # Create Linear X_cond based on feature importance, correlation Shap
    
    X_train, X_test, y_train, y_test = train_test_split_time(X_cond, y_cond, test_frac=0.2)
    
    # print(f"Running Ridge CV")
    # best_alpha_ridge = tune_ridge_alpha_for_config(X_cond, y_cond, frac=0.8, n_splits=5)
    # print(f"Running lasso CV")
    # best_alpha_lasso = tune_lasso_alpha_for_config(X_cond, y_cond, frac=0.8, n_splits=5)
    # print(f"Running ElsticNet CV")
    # best_enet = tune_elasticnet_for_config(X_cond, y_cond)

    if drop_features_bool == True:
        elasticnet_top_features = findTopCorrFeaturesElastic(X_train, y_train, best_enet, corr_key)
        X_train_red = X_train[elasticnet_top_features].copy()
        X_test_red  = X_test[elasticnet_top_features].copy()


    # print(f"Running LGBM GridSearch CV")
    # best_lgbm = tune_lgbm_with_gridsearch(X_train, y_train, frac=0.8, n_splits=2)
    print(f"Running XGBoost GridSearch CV")
    best_xgb = tune_xgb_with_gridsearch(X_train, y_train, frac=0.8, n_splits=5)
    X_train_gpu = cp.asarray(X_train.values)
    X_test_gpu  = cp.asarray(X_test.values)


    if drop_features_bool == True:
        xgb_top_features = findTopImporFeaturesXGB(X_train, y_train, best_xgb, impor_key)
        X_train_blue = X_train[xgb_top_features].copy()
        X_test_blue = X_test[xgb_top_features].copy()
        
    print("Initialising Model Configs")
    models_config = {
        # "ols": make_ols(),
        # "ridge": make_ridge(alpha=best_alpha_ridge),
        # "lasso":      make_lasso(alpha=best_alpha_lasso),
        # "elasticnet": make_elasticnet(alpha=best_enet["alpha"], l1_ratio=best_enet["l1_ratio"]),
        # "lgbm":       make_lgbm(params=best_lgbm),
        "xgb":        make_xgb(params=best_xgb)
    }
    for model_name, model in tqdm(
        models_config.items(),
        desc=f"Models ({impute_key}, drop={drop_level}, feat={feat_key})",
        leave=False
    ):
        
        if (drop_features_bool == True):
            X_train_eval = []
            X_test_eval = []
            if (model_name == "elasticnet" or model_name == "lasso"):
                X_train_eval = X_train_red
                X_test_eval = X_test_red
            else:
                X_train_eval = X_train_blue
                X_test_eval = X_test_blue
    
            # Standard CV
            print(f"Evaluating model {model_name}")
            holdout_metrics = eval_holdout(model, X_train_eval, y_train, X_test_eval, y_test)
        else:
            print(f"Evaluating model {model_name}")
            holdout_metrics = eval_holdout(model, X_train_gpu, y_train, X_test_gpu, y_test)
        print(holdout_metrics)
        results_table.append({
            "model": model_name,
            "impute": impute_key,
            "drop_level": drop_level,
            "features": feat_key,
            "corr_key": corr_key,
            "impor_key": impor_key,
            "drop_features": drop_features_bool,
            "mode": "standard_cv",
            # "best_alpha_ridge": best_alpha_ridge,
            # "best_alpha_lasso": best_alpha_lasso,
            # "best_enet": best_enet,
            # "best_lgbm": best_lgbm,
            "best_xgb": best_xgb,
            # **std_metrics,
            **holdout_metrics
        })

        # # Optuna Sharpe tuning only for tree models (or only on best few combos)
        # if model_name in ("xgb", "lgbm") and feat_key != "none":
        #     study = tune_xgb_with_optuna(X_cond, y_cond, df_cond, n_trials=20)
        #     best_val = study.best_value
        #     results_table.append({
        #         "model": model_name,
        #         "impute": impute_key,
        #         "features": feat_key,
        #         "mode": "wf_optuna_sharpe",
        #         "adj_sharpe_best": best_val,
        #     })

Configs:   0%|          | 0/1 [00:00<?, ?it/s]

Building X y with conditions -> impute: medium drop: 3 features: medium, drop_features: False
Running XGBoost GridSearch CV
Fitting 5 folds for each of 3 candidates, totalling 15 fits
[CV 1/5] END colsample_bytree=0.6, learning_rate=0.005, max_depth=7, max_leaves=32, min_child_weight=4, n_estimators=4000, subsample=0.7;, score=-0.000 total time=  59.5s
[CV 2/5] END colsample_bytree=0.6, learning_rate=0.005, max_depth=7, max_leaves=32, min_child_weight=4, n_estimators=4000, subsample=0.7;, score=-0.000 total time= 1.4min
[CV 3/5] END colsample_bytree=0.6, learning_rate=0.005, max_depth=7, max_leaves=32, min_child_weight=4, n_estimators=4000, subsample=0.7;, score=-0.000 total time= 1.5min
[CV 4/5] END colsample_bytree=0.6, learning_rate=0.005, max_depth=7, max_leaves=32, min_child_weight=4, n_estimators=4000, subsample=0.7;, score=-0.000 total time= 1.6min
[CV 5/5] END colsample_bytree=0.6, learning_rate=0.005, max_depth=7, max_leaves=32, min_child_weight=4, n_estimators=4000, subsample

Evaluating model xgb


Configs: 100%|██████████| 1/1 [30:16<00:00, 1816.29s/it]

{'r2_insample': 0.9585258859780743, 'r2_holdout': -0.12300173695989614}


## Imputation Pipeline Step

- fill 0
- cut off row

# Run Optuna Function
- CV functions vs Optuna Adjusted Sharpe Optimised.
- LassoCV
- RidgeCV
- Elastic-Net
- LGBM w/ Grid Search CV
- XGBoost w/ Grid Search CV

Against
Optuna Adjusted Sharpe Optimised tuning function.


## Table of Results